<a href="https://colab.research.google.com/github/asta3002/FISH/blob/main/Custom_TensorNEAT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Installation

In [1]:
# !pip install git+https://github.com/EMI-Group/tensorneat.git
!pip install git+https://github.com/asta3002/Custom_tensorneat_backprop.git

  Cloning https://github.com/asta3002/Custom_tensorneat_backprop.git to /tmp/pip-req-build-5230ylu5
  Running command git clone --filter=blob:none --quiet https://github.com/asta3002/Custom_tensorneat_backprop.git /tmp/pip-req-build-5230ylu5
  Resolved https://github.com/asta3002/Custom_tensorneat_backprop.git to commit 3013cf5858053d5d8f2960b3fd9f5047a1717e91
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.1/344.1 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.6/86.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.4/172.4 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 98.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!pip install -U "jax[tpu]==0.4.28"

INFO: pip is looking at multiple versions of jax[tpu] to determine which version is compatible with other requirements. This could take a while.
ERROR: Ignored the following yanked versions: 0.0.1
ERROR: Could not find a version that satisfies the requirement libtpu-nightly==0.1.dev20240508; extra == "tpu" (from jax[tpu]) (from versions: none)
ERROR: No matching distribution found for libtpu-nightly==0.1.dev20240508; extra == "tpu"


In [3]:
!pip install evojax

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.1/96.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.8/303.8 kB 18.1 MB/s eta 0:00:00


#Libraries

In [4]:
# Import necessary modules
import jax
import jax.numpy as jnp
import argparse
import os
from jax import random
from evojax.task.slimevolley import SlimeVolley
from tensorneat.problem import BaseProblem
from tensorneat.pipeline import Pipeline
from tensorneat.algorithm.neat import NEAT
from tensorneat.genome import DefaultGenome, BiasNode, DefaultConn,DefaultMutation,OriginConn,OriginNode
from tensorneat.genome.operations import DefaultCrossover, DefaultDistance, DefaultMutation
from tensorneat.common import ACT, AGG
from typing import Tuple
from functools import partial

#DEMO

In [ ]:
max_steps = 3000
task = SlimeVolley(test=True, max_steps=max_steps)
task_reset_fn = jax.jit(task.reset)
task_step_fn = jax.jit(task.step)
og_key = jax.random.PRNGKey(12346)[None,:]
task_state = task_reset_fn(og_key)
test_state = task.reset(og_key)
# dummy_action = jnp.array([[0.0, 1.0, -1.0]], dtype=jnp.float32)  # shape [1, 3]
# # next_state,reward,done = task_step_fn(task_state,dummy_action)
# # action = (network_output > 0.0).astype(jnp.int32)


In [ ]:
print(test_state.game_state.agent_right)

AgentState(direction=Array([1], dtype=int32, weak_type=True), x=Array([12.], dtype=float32, weak_type=True), y=Array([1.5], dtype=float32, weak_type=True), r=Array([1.5], dtype=float32, weak_type=True), vx=Array([0], dtype=int32, weak_type=True), vy=Array([0], dtype=int32, weak_type=True), desired_vx=Array([0], dtype=int32, weak_type=True), desired_vy=Array([0], dtype=int32, weak_type=True), life=Array([5], dtype=int32, weak_type=True))


In [ ]:
print(test_state.game_state.agent_left)

AgentState(direction=Array([-1], dtype=int32, weak_type=True), x=Array([-12.], dtype=float32, weak_type=True), y=Array([1.5], dtype=float32, weak_type=True), r=Array([1.5], dtype=float32, weak_type=True), vx=Array([0], dtype=int32, weak_type=True), vy=Array([0], dtype=int32, weak_type=True), desired_vx=Array([0], dtype=int32, weak_type=True), desired_vy=Array([0], dtype=int32, weak_type=True), life=Array([5], dtype=int32, weak_type=True))


In [ ]:
print(test_state.game_state.ball)

ParticleState(x=Array([0.], dtype=float32), y=Array([12.], dtype=float32), prev_x=Array([0.], dtype=float32), prev_y=Array([12.], dtype=float32), vx=Array([6.860523], dtype=float32), vy=Array([20.072697], dtype=float32), r=Array([0.5], dtype=float32))


In [ ]:
def squeeze_state(state):
        return jax.tree_util.tree_map(lambda x: jnp.squeeze(x, axis=0) if hasattr(x, "ndim") and x.ndim > 0 else x, state)

In [ ]:
# task.render(squeeze_state(test_state))

In [ ]:
print(task_state.obs)

[[1.2       0.15      0.        0.        0.        1.2       0.6860523
  2.0072696 1.2       0.15      0.        0.       ]]


In [ ]:
print(task_state.game_state.agent_left)

AgentState(direction=Array([-1], dtype=int32, weak_type=True), x=Array([-12.], dtype=float32, weak_type=True), y=Array([1.5], dtype=float32, weak_type=True), r=Array([1.5], dtype=float32, weak_type=True), vx=Array([0], dtype=int32, weak_type=True), vy=Array([0], dtype=int32, weak_type=True), desired_vx=Array([0], dtype=int32, weak_type=True), desired_vy=Array([0], dtype=int32, weak_type=True), life=Array([5], dtype=int32, weak_type=True))


In [ ]:
print(task_state.game_state.agent_right)

AgentState(direction=Array([1], dtype=int32, weak_type=True), x=Array([12.], dtype=float32, weak_type=True), y=Array([1.5], dtype=float32, weak_type=True), r=Array([1.5], dtype=float32, weak_type=True), vx=Array([0], dtype=int32, weak_type=True), vy=Array([0], dtype=int32, weak_type=True), desired_vx=Array([0], dtype=int32, weak_type=True), desired_vy=Array([0], dtype=int32, weak_type=True), life=Array([5], dtype=int32, weak_type=True))


In [ ]:
print(task_state.game_state.ball)

ParticleState(x=Array([0.], dtype=float32), y=Array([12.], dtype=float32), prev_x=Array([0.], dtype=float32), prev_y=Array([12.], dtype=float32), vx=Array([6.860523], dtype=float32), vy=Array([20.072697], dtype=float32), r=Array([0.5], dtype=float32))


In [ ]:
# print(task_state.game_state.agent_left.life,task_state.game_state.agent_right.life)

In [ ]:
# next_state = task_state

In [ ]:
# total_reward = jnp.array(0.0, dtype=jnp.float32)

In [ ]:
# for i in range(3000):
#   next_state,reward,done = task_step_fn(next_state,dummy_action)
#   reward = jnp.squeeze(reward).astype(jnp.float32)
#   if(reward):
#     print(reward,done,next_state.game_state.agent_left.life,next_state.game_state.agent_right.life)
#   total_reward = total_reward + reward
#   if done:
#     break




In [ ]:
# print(total_reward)

In [ ]:
# print(task_state.game_state)
# print(task_state.obs)

#SlimeVolleyBall Environment

In [5]:
import jax.numpy as jnp

def adjusted_fitness(s_fitness):
    """
    Calculates the total adjusted fitness for a species, as described in the original NEAT paper.

    Args:
        s_fitness (jnp.ndarray): A JAX array containing the fitness values of individuals
                                 in a single species. Non-members are represented by -jnp.inf.

    Returns:
        jnp.ndarray: The total adjusted fitness of the species.
    """
    # 1. Create a boolean mask to identify valid fitness values (i.e., not -jnp.inf)
    is_member_mask = jnp.isfinite(s_fitness)

    # 2. Count the number of members in the species
    species_size = jnp.sum(is_member_mask)

    # Use a conditional to handle the case of an empty species (size=0) to avoid division by zero.
    species_size = jnp.where(species_size == 0, 1, species_size)

    # 3. Adjust the fitness of each member by dividing by the species size.
    # The where clause ensures only valid fitnesses are adjusted.
    adjusted_s_fitness = jnp.where(is_member_mask, s_fitness / species_size, 0.0)

    # 4. Sum the adjusted fitness values to get the species' total adjusted fitness.
    total_adjusted_fitness = jnp.sum(adjusted_s_fitness)

    return total_adjusted_fitness

In [6]:
VELOCITY_THRESHOLD = 15.0

In [15]:
class SlimeVolleyEnv(BaseProblem):
    jitable = True  # Needed for TensorNEAT JIT support

    def __init__(self, max_episode_steps=3000, num_episodes=3):
        super().__init__()
        self.max_steps = max_episode_steps
        self.episodes = num_episodes

        # Create task instance
        self.task = SlimeVolley(test=True, max_steps=self.max_steps) #To imitate the same conditions as testing phase.
        self.test_task = SlimeVolley(test=True, max_steps=3000)

        # JIT compile reset and step functions
        self.task_reset = jax.jit(self.task.reset)
        self.task_step = jax.jit(self.task.step)
        self.test_task_reset = jax.jit(self.test_task.reset)
        self.test_task_step = jax.jit(self.test_task.step)

    def calculate_episode_reward(self,final_state):
        """
        Improved reward function with progressive life loss penalties
        """
        # max_steps = 3000.0
        # opp_life = final_state.game_state.agent_left.life
        # agent_life = final_state.game_state.agent_right.life
        # opp_lives_lost = 5 - opp_life
        # agent_lives_lost = 5 - agent_life
        # steps_taken = final_state.steps

        # # Base reward: Heavy emphasis on life differential
        # life_differential = (opp_lives_lost - agent_lives_lost) * 100.0

        # # Damage dealing reward (encourage any damage)
        # damage_bonus = opp_lives_lost * 75.0

        # # NEW: Progressive penalty for each life lost by our agent
        # # Each life lost costs 100 points - discourages reckless play
        # life_loss_penalty = agent_lives_lost * -100.0

        # # Survival reward (staying alive matters)
        # survival_bonus = agent_life * 25.0

        # # CRITICAL: Massive defeat penalty (complete loss)
        # defeat_penalty = jnp.where(agent_life == 0, -400.0, 0.0)

        # # Victory bonus (must be earned)
        # victory_bonus = jnp.where(opp_life == 0, 600.0, 0.0)

        # # Time efficiency (only for victories)
        # time_efficiency = jnp.where(
        #     opp_life == 0,
        #     ((max_steps - steps_taken) / max_steps) * 150.0,
        #     0.0
        # )

        # # Small bonus for just surviving longer (exploration reward)
        # survival_time = (steps_taken / max_steps) * 10.0

        # total_reward = (
        #     life_differential +
        #     damage_bonus +
        #     life_loss_penalty +  # NEW: Penalty per life lost
        #     survival_bonus +
        #     defeat_penalty +  # Additional penalty for total defeat
        #     victory_bonus +
        #     time_efficiency +
        #     survival_time
        # )
        opp_life = final_state.game_state.agent_left.life
        agent_life = final_state.game_state.agent_right.life
        total_reward= ((5-opp_life)*2)- (5-agent_life)

        return total_reward




    def evaluate(self, state, randkey, act_func, params):
        """Evaluate with episode-end reward calculation, maintaining JAX parallelization"""

        def run_episode(ep_key):
            # Reset environment
            reset_key = ep_key[None, :]
            task_state = self.task_reset(reset_key)

            # Initialize carry with episode_ended flag
            episode_ended = jnp.array(False, dtype=jnp.bool_)
            init_carry = (task_state, episode_ended)

            def step_fn(carry, _):
                task_state, ended = carry

                # Get observation and action
                obs = task_state.obs.squeeze()
                action_out = act_func(task_state, params, obs)
                action = (action_out > 0.0).astype(jnp.float32)
                jax_action = action[None,:]

                # Step environment
                new_state, reward, done = self.task_step(task_state, jax_action)
                done = jnp.squeeze(done).astype(jnp.bool_)

                # CRITICAL: Use previous 'ended' flag for freezing decision
                # This ensures we capture the state when done=True fires
                final_state = jax.tree_util.tree_map(
                    lambda old, new: jnp.where(ended, old, new),
                    task_state, new_state
                )

                # Update ended flag AFTER freezing decision
                new_ended = ended | done

                return (final_state, new_ended), None

            # Run episode for max_steps (state freezes after done=True)
            final_carry, _ = jax.lax.scan(step_fn, init_carry, None, length=self.max_steps)
            final_state, _ = final_carry

            # Calculate episode reward based on final outcome
            episode_reward = self.calculate_episode_reward(final_state)

            return episode_reward

        # Run multiple episodes in parallel using vmap
        ep_keys = jax.random.split(randkey, self.episodes)
        rewards = jax.vmap(run_episode)(ep_keys)

        return jnp.mean(rewards)

    @property
    def input_shape(self):
        return self.task.obs_shape  # Provided by SlimeVolley

    @property
    def output_shape(self):
        return self.task.act_shape  # Provided by SlimeVolley

    def squeeze_state(self, state):
        return jax.tree_util.tree_map(
            lambda x: jnp.squeeze(x, axis=0) if hasattr(x, "ndim") and x.ndim > 0 else x,
            state
        )

    def show(self, state, randkey, act_func, params, *args, **kwargs):
        """Render a few episodes to visually inspect performance."""
        import time
        import numpy as np
        print("\n" + "="*50)
        print("SLIMEVOLLEY AGENT PERFORMANCE")
        print("="*50)

        best_reward = float('-inf')
        best_screens = []
        best_opp_life = float('inf')  # Track opponent life (lower is better)
        best_steps = 0  # Track episode length (higher is better)

        # JAX-compiled function for running a single episode
        @jax.jit
        def run_single_episode(episode_key):
            """Run a single episode and return states, rewards, and done flags."""
            test_task_state = self.test_task_reset(episode_key[None, :])

            def step_fn(carry, _):
                task_state, total_reward, step_count, terminated = carry

                obs = task_state.obs.squeeze()
                action_out = act_func(task_state, params, obs)
                action = (action_out > 0.0).astype(jnp.float32)
                jax_action = action[None, :]

                new_state, reward, done = self.test_task_step(task_state, jax_action)
                reward = jnp.squeeze(reward).astype(jnp.float32)
                done = jnp.squeeze(done).astype(jnp.bool_)

                # Update only if not already terminated
                new_total_reward = jnp.where(terminated, total_reward, total_reward + reward)
                new_step_count = jnp.where(terminated, step_count, step_count + 1)
                new_terminated = terminated | done

                # Use previous state if already terminated
                final_state = jax.tree_util.tree_map(
                    lambda old, new: jnp.where(terminated, old, new),
                    task_state, new_state
                )

                return (final_state, new_total_reward, new_step_count, new_terminated), {
                    'state': final_state,
                    'reward': reward,
                    'done': done,
                    'terminated': new_terminated
                }

            init_carry = (test_task_state, jnp.array(0.0, dtype=jnp.float32),
                        jnp.array(0, dtype=jnp.int32), jnp.array(False))

            final_carry, history = jax.lax.scan(step_fn, init_carry, None, length=3000)

            return {
                'final_reward': final_carry[1],
                'final_steps': final_carry[2],
                'states': history['state'],
                'rewards': history['reward'],
                'dones': history['done'],
                'terminated_flags': history['terminated']
            }

        # Run episodes using JAX
        episode_keys = jax.random.split(randkey, 5)

        for i in range(5):
            # Run episode with JAX
            episode_result = run_single_episode(episode_keys[i])

            total_reward = float(episode_result['final_reward'])
            final_steps = int(episode_result['final_steps'])
            states = episode_result['states']
            dones = episode_result['dones']

            # Find the actual end of the episode
            done_indices = jnp.where(dones)[0]
            if len(done_indices) > 0:
                actual_end_step = int(done_indices[0]) + 1
                # agent_left is the opponent, agent_right is our agent
                # Extract life value properly - handle potential array dimensions
                opp_life_array = states.game_state.agent_left.life[done_indices[0]]
                # Squeeze any extra dimensions and convert to scalar
                opp_end_life = float(jnp.squeeze(opp_life_array))
            else:
                actual_end_step = 3000
                opp_end_life = 5.0  # Max life if episode didn't end

            print(f"Episode {i+1} - Reward: {total_reward:.2f}, Steps: {actual_end_step}, Opponent Life: {opp_end_life}")

            # Generate screens for this episode (only up to actual end)
            current_screens = []
            for step in range(actual_end_step):
                step_state = jax.tree_util.tree_map(lambda x: x[step], states)
                render_state = self.squeeze_state(step_state)
                current_screens.append(self.test_task.render(render_state))

            # Determine if this episode is better based on hierarchical criteria
            is_better = False

            if total_reward > best_reward:
                # Higher reward is always better
                is_better = True
            elif total_reward == best_reward:
                # Same reward: prefer lower opponent life
                if opp_end_life < best_opp_life:
                    is_better = True
                elif opp_end_life == best_opp_life:
                    # Same reward and opponent life: prefer longer episode
                    if actual_end_step > best_steps:
                        is_better = True

            if is_better:
                best_reward = total_reward
                best_opp_life = opp_end_life
                best_steps = actual_end_step
                best_screens = current_screens.copy()
                print(f"  -> New best episode!")

        print(f"\n{'='*50}")
        print(f"BEST EPISODE SUMMARY:")
        print(f"  Reward: {best_reward:.2f}")
        print(f"  Opponent Life Remaining: {best_opp_life}")
        print(f"  Episode Length: {best_steps} steps")
        print(f"{'='*50}\n")

        # Create GIF only from the best episode
        log_dir = './'
        gif_file = os.path.join(log_dir, 'slimevolley_best.gif')
        if best_screens:
            best_screens[0].save(gif_file, save_all=True, append_images=best_screens[1:],
                                duration=40, loop=0)
            print(f"GIF created from best episode and saved to: {gif_file}")
        else:
            print("No screens to save for GIF")

#Training

In [10]:
neat_algorithm=NEAT(
        pop_size=5000,  # Larger population for complex task
        species_size=50,  # Maximum number of species allowed.More species for diversity
        survival_threshold=1.0,  #  Fraction of each species (sorted by fitness) that can be selected as parents (0-1). Only the top X% survive to reproduce, creating selection pressure.
        genome=DefaultGenome(
            num_inputs=12,  # SlimeVolley observation space
            num_outputs=3,  # SlimeVolley action space (3 discrete actions)
            max_nodes=100,
            max_conns=1500,
            init_hidden_layers=(),  # Start with no hidden layers
            node_gene=OriginNode( bias_init_mean= 0.0,
                                bias_init_std = 0.1,
                                bias_mutate_power = 0.05,
                                bias_mutate_rate = 0.1,
                                bias_replace_rate = 0.005,
                                bias_lower_bound = -3,
                                bias_upper_bound = 3,
                                aggregation_default = AGG.sum,
                                aggregation_options = [AGG.sum, AGG.mean, AGG.max,AGG.product],
                                aggregation_replace_rate = 0.0,
                                activation_default = ACT.relu,
                                activation_options=[
                                                    ACT.relu,
                                                    ACT.sigmoid,
                                                    ACT.sin,
                                                    ACT.tanh,
                                                    ACT.identity,
                                                    ACT.scaled_sigmoid,
                                                    ACT.scaled_tanh,
                                                    ACT.lelu,
                                                    ACT.inv,
                                                    ACT.abs,
                                                    ACT.gaussian,
                                                    ACT.softplus,
                                                  ],
                                activation_replace_rate= 0.05,


                                ),
            conn_gene=OriginConn(weight_init_mean  = 0.0,
                                  weight_init_std   = 1.0,
                                  weight_mutate_power = 0.2, #scale of perturbation
                                  weight_mutate_rate  = 0.1,  #probability of perturbation
                                  weight_replace_rate = 0.005,
                                  weight_lower_bound = -5,
                                  weight_upper_bound = 5,
                                  ),
            mutation=DefaultMutation(conn_add = 0.3,
                                     conn_delete  = 0.05,
                                     node_add = 0.3,
                                     node_delete = 0.05,
                                     ),
            crossover=DefaultCrossover(),
            distance=DefaultDistance(compatibility_disjoint = 1.0,
                                     compatibility_weight  = 0.4,
                                     ),
            output_transform=ACT.tanh, # Tanh output activation
            input_transform=None,


                        ),

        max_stagnation =  30, #Number of generations a species can go without improvement before being eliminated.
        species_elitism = 7, #Number of top-ranked species protected from stagnation. These species survive even if they haven't improved recently.
        spawn_number_change_rate= 0.5, # Controls how quickly species population sizes can change between generations (0-1). Smooths population changes: new_size = old_size + (target_size - old_size) * rate. Lower values = more gradual changes.
        genome_elitism  = 7, #- Number of best individuals per species that are preserved without mutation. They undergo crossover but skip mutation to preserve top performers.
        min_species_size = 1, #Minimum number of individuals a species must have. Prevents species from becoming too small and going extinct prematurely.
        compatibility_threshold = 1.0, #Maximum genetic distance for an individual to join a species. Lower values = stricter speciation (more species with similar members). Higher values = looser speciation (fewer, more diverse species
        species_fitness_func = jnp.max, #Function to compute species fitness from member fitnesses (default: jnp.max). Could also use mean, median, etc. Determines how species are ranked.
        species_number_calculate_by ="rank",#Method for allocating population across species( "rank": Linear ranking ; "fitness": Proportional to actual fitness values (higher fitness = more members) )
    )


In [11]:
pipeline = Pipeline(
    algorithm=neat_algorithm,
    problem=SlimeVolleyEnv(
        max_episode_steps=3000,  # Shorter episodes for faster training
        num_episodes=3  # Average over 3 episodes per evaluation
    ),
    generation_limit=200,  # Run for 200 generations
    fitness_target=1000.0,  # Stop if this fitness is reached
    seed=23,
)



In [12]:
state = pipeline.setup()

state, best = pipeline.auto_run(state)

initializing
initializing finished
start compile
compile finished, cost time: 39.608932s
Generation: 1, Cost time: 29755.29ms
 	fitness: valid cnt: 5000, max: -333.3333, min: -500.0000, mean: -488.1733, std: 20.4971

	node counts: max: 16, min: 15, mean: 15.29
 	conn counts: max: 38, min: 34, mean: 36.28
 	species: 50, [43, 38, 34, 16, 26, 37, 20, 57, 13, 35, 10, 4, 10, 16, 12, 41, 24, 3, 11, 73, 4, 6, 5, 108, 16, 7, 4, 12, 5, 3, 2, 16, 1, 17, 18, 65, 35, 20, 37, 7, 10, 19, 75, 31, 65, 11, 17, 1, 19, 3841]

Generation: 2, Cost time: 32336.56ms
 	fitness: valid cnt: 5000, max: -333.3333, min: -500.0000, mean: -488.0934, std: 20.6884

	node counts: max: 17, min: 15, mean: 15.46
 	conn counts: max: 40, min: 33, mean: 36.44
 	species: 50, [3, 31, 59, 40, 62, 5, 37, 7, 51, 45, 38, 46, 12, 41, 24, 36, 27, 46, 58, 34, 84, 45, 42, 37, 43, 22, 13, 30, 24, 8, 33, 15, 59, 21, 25, 54, 28, 18, 16, 18, 5, 14, 19, 53, 22, 4, 6, 8, 6, 3526]

Generation: 3, Cost time: 33968.14ms
 	fitness: valid cnt: 5

In [13]:
# show result
pipeline.show(state, best)


SLIMEVOLLEY AGENT PERFORMANCE
Episode 1 - Reward: -5.00, Steps: 367, Opponent Life: 5.0
  -> New best episode!
Episode 2 - Reward: -4.00, Steps: 574, Opponent Life: 4.0
  -> New best episode!
Episode 3 - Reward: -5.00, Steps: 373, Opponent Life: 5.0
Episode 4 - Reward: -4.00, Steps: 421, Opponent Life: 4.0
Episode 5 - Reward: -5.00, Steps: 367, Opponent Life: 5.0

BEST EPISODE SUMMARY:
  Reward: -4.00
  Opponent Life Remaining: 4.0
  Episode Length: 574 steps

GIF created from best episode and saved to: ./slimevolley_best.gif


In [14]:
network = neat_algorithm.genome.network_dict(state, *best)
neat_algorithm.genome.visualize(network, save_path="./volleyball_network.png")

In [ ]:

github_path = "First/NEAT_JAX/Custom_TensorNEAT.ipynb"